In [ ]:
import requests
import json

# Replace this with your actual JSON URL
url_json = "https://rest.uniprot.org/uniprotkb/stream?format=json&query=%28%28length%3A%5B40+TO+*%5D%29+AND+%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28+%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0188%29+OR+%28cc_scl_term%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0162%29+%29+NOT+%28ft_signal%3A*%29%29&size=50"



# 1. Fetch the data from the URL
response = requests.get(url_json)

# 2. Parse the response into a Python dictionary
data = response.json()

# 3. Pretty-print the JSON data with a 4-space indent
print(json.dumps(data, indent=4))

ChunkedEncodingError: Response ended prematurely

In [ ]:
import pandas as pd

url = "https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cft_transmem%2Cft_helix&format=tsv&query=accession%3AA3KMH1+OR+accession%3AO75874+OR+accession%3AQ6YN16+OR+accession%3AQ9UHK6"

# Read directly from the URL, handling the gzip compression automatically
df = pd.read_csv(url, sep='\t', compression='gzip')

# Display the first few rows
df.head()

,Entry,Reviewed,Entry Name,Protein names,Transmembrane,Helix
0,A3KMH1,reviewed,VWA8_HUMAN,von Willebrand factor A domain-containing prot...,NaN,NaN
1,O75874,reviewed,IDHC_HUMAN,Isocitrate dehydrogenase [NADP] cytoplasmic (I...,NaN,"HELIX 17..26; /evidence=""ECO:0007829|PDB:6BKX""..."
2,Q6YN16,reviewed,HSDL2_HUMAN,Hydroxysteroid dehydrogenase-like protein 2 (E...,NaN,"HELIX 21..31; /evidence=""ECO:0007829|PDB:3KVO""..."
3,Q9UHK6,reviewed,AMACR_HUMAN,Alpha-methylacyl-CoA racemase (EC 5.1.99.4) (2...,NaN,NaN


In [ ]:
import pandas as pd

url = "https://rest.uniprot.org/uniprotkb/search?fields=accession%2Cid%2Corganism_name%2Clineage%2Clength%2Cfeature_count%2Cft_helix%2Cft_transmem&format=tsv&query=%28%28length%3A%5B40+TO+*%5D%29+AND+%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28+%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0188%29+OR+%28cc_scl_term%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0162%29+%29+NOT+%28ft_signal%3A*%29%29&size=50"

# Read directly from the URL, handling the gzip compression automatically
df = pd.read_csv(url, sep='\t')

# Display the first few rows
df.head()

,Entry,Entry Name,Organism,Taxonomic lineage,Length,Features,Helix,Transmembrane
0,Q6YN16,HSDL2_HUMAN,Homo sapiens (Human),"cellular organisms (no rank), Eukaryota (domai...",418,Active site (1); Alternative sequence (1); Bet...,"HELIX 21..31; /evidence=""ECO:0007829|PDB:3KVO""...",NaN
1,Q9UHK6,AMACR_HUMAN,Homo sapiens (Human),"cellular organisms (no rank), Eukaryota (domai...",382,Active site (2); Alternative sequence (5); Bin...,NaN,NaN
2,A3KMH1,VWA8_HUMAN,Homo sapiens (Human),"cellular organisms (no rank), Eukaryota (domai...",1905,Alternative sequence (2); Binding site (1); Ch...,NaN,NaN
3,O75874,IDHC_HUMAN,Homo sapiens (Human),"cellular organisms (no rank), Eukaryota (domai...",414,Beta strand (19); Binding site (13); Chain (1)...,"HELIX 17..26; /evidence=""ECO:0007829|PDB:6BKX""...",NaN
4,O75521,ECI2_HUMAN,Homo sapiens (Human),"cellular organisms (no rank), Eukaryota (domai...",394,Alternative sequence (1); Beta strand (8); Bin...,"HELIX 40..52; /evidence=""ECO:0007829|PDB:2CQU""...",NaN


# Batch Approach

In [ ]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re

In [ ]:

url = "https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28length%3A%5B40+TO+*%5D%29+AND+%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28+%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0188%29+OR+%28cc_scl_term%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0162%29+%29+NOT+%28ft_signal%3A*%29%29"

batch_size = 500

In [ ]:
res = requests.get(url)
data = res.json()["results"]
len(data)

200

In [ ]:
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

In [ ]:
def get_next_link(headers):
    if "Link" in headers:
        # The regular expression is used to extract the next link for pagination
        re_next_link = re.compile(r'<(.+)>; rel="next"')
        match = re_next_link.match(headers["Link"])
        if match:
            return match.group(1)

# This function actually retrieve the next data batch in the search.
# The function act as an iterator, yielding the next result batch at every call
# The function terminates after the last batch has been returned. In this case,
# the next link will be None
def get_batch(batch_url):
    while batch_url:
        # Run the API call
        response = session.get(batch_url)
        # Will raise an error if an error status code is obtained
        response.raise_for_status()
        # Get the total number of entries in the search
        total = response.headers["x-total-results"]
        # Yield the response and the total number of entries
        yield response, total
        # Get the link to the API call for the next data batch
        batch_url = get_next_link(response.headers)

In [ ]:
def extract_fields(entry):
    # We extract the accession, the sequence length and the start and end location of the first coiled-coil segment
    s, e = 0, 0

    accession = entry.get("primaryAccession", "")
    organism_name = entry.get("organism", {}).get("scientificName", "")
    length = entry.get("sequence", {}).get("length", 0)

    #kindom
    lineage = entry.get("organism", {}).get("lineage", [])
    if "Metazoa" in lineage:
        kingdom = "Metazoa"
    elif "Fungi" in lineage:
        kingdom = "Fungi"
    elif "Viridiplantae" in lineage:
        kingdom = "Plants"
    else:
        kingdom = "Other"

    early_trans = False

    for f in entry.get("features", []):
        if f.get("type") == "Transmembrane" and f.get("description") == "Helical":
            try:
                start_val = f["location"]["start"]["value"]
                if start_val < 90:
                    early_trans = True
                    break
            except KeyError:

                continue

    return (accession, organism_name, kingdom, length, early_trans)

In [ ]:

def get_dataset(search_url, extract_function, output_file_name):
    filtered_json = []
    n_total, n_filtered = 0, 0

    # Run the API call in batches
    for batch, total in get_batch(search_url):
        # parse the JSON body of the response
        batch_json = json.loads(batch.text)


        for entry in batch_json.get("results", []):
            n_total += 1
            filtered_json.append(extract_function(entry))
            n_filtered += 1

    print(n_total, n_filtered)

    with open(output_file_name, "w") as ofs:
        for entry in filtered_json:
            # Print the fields in TSV format
            print(*entry, sep="\t", file=ofs)

In [ ]:
output_file = "dataset_neg.tsv"
get_dataset(url, extract_fields, output_file)

KeyboardInterrupt: 

# dowload fasta intero

In [ ]:
url_fasta=url = "https://rest.uniprot.org/uniprotkb/search?format=fasta&query=%28%28length%3A%5B40+TO+*%5D%29+AND+%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28+%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0188%29+OR+%28cc_scl_term%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0162%29+%29+NOT+%28ft_signal%3A*%29%29"
!wget url_fasta